# M2 Kelompok 03
Kode diambil dari branch `eksperimen-m2` di GitHub


In [ ]:
import shutil, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
print('Python:', sys.version)
available_kb = int(next(line.split()[1] for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemAvailable:')))
print('MemAvailable (GB):', round(available_kb / 1e6, 2))
print('Free disk (GB):', round(shutil.disk_usage('/content').free / 1e9, 2))
assert available_kb >= 6_000_000, 'RAM tersedia kurang dari 6 GB; jangan jalankan encoder'
assert shutil.disk_usage('/content').free >= 10_000_000_000, 'Ruang Colab kurang dari 10 GB'


In [ ]:
repo = Path('/content/Tubes-Multimodal')
assert not repo.exists(), 'Gunakan runtime Colab baru agar repo bersih; jangan menimpa folder yang sudah ada.'
subprocess.run(['git', 'clone', '--branch', 'eksperimen-m2', '--single-branch', 'https://github.com/bayybrigas04/Tubes-Multimodal.git', str(repo)], check=True)
branch = subprocess.check_output(['git', '-C', str(repo), 'branch', '--show-current'], text=True).strip()
assert branch == 'eksperimen-m2', f'Branch keliru: {branch}'
assert (repo / 'src' / 'M2_Encoders.py').is_file(), 'Kode M2 tidak ditemukan pada branch eksperimen-m2'
print('Commit M2:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo / 'requirements.txt'), 'open_clip_torch', 'torchvision'], check=True)


In [ ]:
zip_path = Path('/content/drive/MyDrive/Dataset-Tubes-Multimodal/bundle-mahasiswa.zip')
assert zip_path.is_file(), f'ZIP tidak ada: {zip_path}'
data_root = Path('/content/data')
subprocess.run(['unzip', '-q', str(zip_path), '-d', str(data_root)], check=True)
data_dir = data_root / 'bundle-mahasiswa'
assert all((data_dir / filename).is_file() for filename in ('train.csv', 'val.csv')), f'Struktur ZIP tidak sesuai: {data_dir}'
assert (data_dir / 'images').is_dir()
print('Dataset:', data_dir)


## Gate sebelum E1

In [ ]:
import datetime
run_name = 'e1-val-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_dir = Path('/content/drive/MyDrive/Dataset-Tubes-Multimodal/m2-runs') / run_name
assert not out_dir.exists()
command = [sys.executable, str(repo / 'src' / 'M2_Encoders.py'), '--data', str(data_dir), '--out', str(out_dir), '--baseline', str(repo / 'results/m1/tabel_recall_val.csv'), '--e1-revision', '75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2', '--e2-revision', 'a7c4a9bf24872c73a2c059ab2f9e9123a4d4144e', '--models', 'E1', '--skip-projection', '--epochs', '5', '--lr', '0.001', '--batch-size', '4', '--seed', '42']
print('Output:', out_dir)
subprocess.run(command, check=True)
print((out_dir / 'm2_val.csv').read_text(encoding='utf-8'))


## E2 — NLLB-CLIP-SigLIP tanpa proyeksi
Jalankan setelah E1 diperiksa.


In [ ]:
import datetime
import subprocess
import sys
from collections import deque
from pathlib import Path
assert (repo / 'src' / 'M2_Encoders.py').is_file(), 'Kode M2 belum tersedia'
assert (data_dir / 'val.csv').is_file(), 'Dataset val belum tersedia'
runs_dir = Path('/content/drive/MyDrive/Dataset-Tubes-Multimodal/m2-runs')
runs_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
e2_out = runs_dir / f'e2-val-{stamp}'
e2_log = runs_dir / f'e2-val-{stamp}.log'
assert not e2_out.exists() and not e2_log.exists(), 'Nama run sudah terpakai'
e2_command = [sys.executable, str(repo / 'src' / 'M2_Encoders.py'), '--data', str(data_dir), '--out', str(e2_out), '--baseline', str(repo / 'results/m1/tabel_recall_val.csv'), '--e1-revision', '75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2', '--e2-revision', 'a7c4a9bf24872c73a2c059ab2f9e9123a4d4144e', '--models', 'E2', '--skip-projection', '--epochs', '5', '--lr', '0.001', '--batch-size', '4', '--seed', '42']
print('Hasil E2:', e2_out)
print('Log E2:', e2_log)
with e2_log.open('w', encoding='utf-8') as log:
    result = subprocess.run(e2_command, stdout=log, stderr=subprocess.STDOUT, check=False, text=True)
print('Exit code:', result.returncode)
with e2_log.open(encoding='utf-8', errors='replace') as log:
    print(''.join(deque(log, maxlen=60))[-8000:])
if result.returncode != 0:
    raise RuntimeError('E2 gagal')
print((e2_out / 'm2_val.csv').read_text(encoding='utf-8'))


## Ablasi E1 — proyeksi linear dan MLP

In [ ]:
import datetime
import subprocess
import sys
from collections import deque
from pathlib import Path
assert (repo / 'src' / 'M2_Encoders.py').is_file(), 'Kode M2 belum tersedia'
assert (data_dir / 'train.csv').is_file() and (data_dir / 'val.csv').is_file(), 'Dataset belum siap'
projection_epochs = 5
projection_lr = 0.001
projection_batch_size = 4
projection_seed = 42
runs_dir = Path('/content/drive/MyDrive/Dataset-Tubes-Multimodal/m2-runs')
runs_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
projection_out = runs_dir / f'e1-projection-{stamp}'
projection_log = runs_dir / f'e1-projection-{stamp}.log'
assert not projection_out.exists() and not projection_log.exists(), 'Nama run sudah terpakai'
projection_command = [
    sys.executable, str(repo / 'src' / 'M2_Encoders.py'),
    '--data', str(data_dir), '--out', str(projection_out),
    '--baseline', str(repo / 'results/m1/tabel_recall_val.csv'),
    '--e1-revision', '75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2',
    '--e2-revision', 'a7c4a9bf24872c73a2c059ab2f9e9123a4d4144e',
    '--models', 'E1',
    '--epochs', str(projection_epochs), '--lr', str(projection_lr),
    '--batch-size', str(projection_batch_size), '--seed', str(projection_seed),
]
print('Output proyeksi:', projection_out)
print('Log proyeksi:', projection_log)
with projection_log.open('w', encoding='utf-8') as log:
    projection_result = subprocess.run(
        projection_command, stdout=log, stderr=subprocess.STDOUT, check=False, text=True
    )
print('Exit code:', projection_result.returncode)
with projection_log.open(encoding='utf-8', errors='replace') as log:
    print(''.join(deque(log, maxlen=60))[-8000:])
if projection_result.returncode != 0:
    raise RuntimeError('Run proyeksi gagal; simpan log dan jangan ulang sebelum diagnosis')
print((projection_out / 'm2_val.csv').read_text(encoding='utf-8'))
print((projection_out / 'costs.json').read_text(encoding='utf-8'))
